# FPL Points Prediction — Colab Runner

The retraining does not fit on an 8 GB laptop: the feature-engineering step expands
~230,000 rows to ~230 columns, and the models need `xgboost` and `lightgbm`. This
notebook runs the whole fixed pipeline on Colab instead.

**Runtime → Change runtime type → High-RAM** if your account offers it. A GPU is not
useful here — scikit-learn, XGBoost and LightGBM all run on CPU for this workload, so
pick a **CPU** runtime with as much RAM as you can get.

| stage | script | roughly |
| --- | --- | --- |
| 1. repair 2024-25 merged_gw | `rebuild_merged_gw.py` | seconds |
| 2. rebuild the dataset | `build_dataset.py` | 2–5 min |
| 3. feature engineering | `build_features.py` | 10–25 min, RAM-hungry |
| 4. train + save models | `train.py` | 30–90 min |

Each script is safe to re-run, and each stage writes its output to disk, so you can
stop after any stage and pick up later.

## 0. Runtime check

In [ ]:
import multiprocessing, os, shutil, sys, platform

try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
except ImportError:
    ram = float(os.popen("awk '/MemTotal/ {print $2}' /proc/meminfo").read() or 0) / 1e6

print(f"python {sys.version.split()[0]} on {platform.platform()}")
print(f"CPUs:  {multiprocessing.cpu_count()}")
print(f"RAM:   {ram:.1f} GB")
print(f"disk:  {shutil.disk_usage('/').free / 1e9:.0f} GB free")

if ram < 11:
    print("\nWARNING: under ~11 GB. Stage 3 may be killed.")
    print("Runtime -> Change runtime type -> High-RAM, or run stage 3 with fewer seasons.")

## 1. Get the project into Colab

Before running this: upload **`fpl_colab.zip`** to your Google Drive. Drop it in the
top level of *My Drive* — that is what the path below expects.

The zip is 21 MB and holds the 77 files this pipeline reads. Do **not** copy the whole
project folder to Drive: it is 17,368 files, 13,228 of which are per-player CSVs that
nothing here opens, and Drive is very slow at that file count.

The cell below mounts Drive, unzips into `/content/fpl` (local disk, much faster to
read than Drive), and moves into it. Section 8 copies the results back to Drive at the
end, so nothing is lost when the runtime shuts down.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/fpl_colab.zip'   # <-- where you put the zip
PROJECT = '/content/fpl'

import os
assert os.path.exists(ZIP), (
    f"{ZIP} not found. Upload fpl_colab.zip to the top level of My Drive, "
    f"or edit ZIP to wherever you put it."
)

!rm -rf $PROJECT
!mkdir -p $PROJECT
!unzip -q -o "$ZIP" -d $PROJECT

%cd $PROJECT
!ls

In [ ]:
import os

for required in ('final.ipynb', 'scripts', 'data', 'all_seasons_data_final.csv'):
    assert os.path.exists(required), (
        f"{required!r} missing from {os.getcwd()} -- the unzip did not produce "
        f"the expected layout."
    )

seasons = sorted(d for d in os.listdir('data') if d[0].isdigit())
print(f"project root: {os.getcwd()}")
print(f"seasons:      {seasons}")
print(f"scripts:      {sorted(os.listdir('scripts'))}")

<details>
<summary>Alternative: work directly inside Drive instead</summary>

Slower to read and write, but `saved_models/` survives a runtime shutdown without the
copy-back step. Unzip into a Drive folder and point `PROJECT` at it:

```python
PROJECT = '/content/drive/MyDrive/fpl'
!mkdir -p $PROJECT && unzip -q -o /content/drive/MyDrive/fpl_colab.zip -d $PROJECT
%cd $PROJECT
```
</details>

## 2. Dependencies

In [ ]:
!pip install -q xgboost lightgbm pulp
import xgboost, lightgbm, sklearn, pandas, numpy
for m in (pandas, numpy, sklearn, xgboost, lightgbm):
    print(f"{m.__name__:<12} {m.__version__}")

In [ ]:
# A `!command` that fails does NOT stop Colab's "Run all" -- it just prints the
# error and carries on into the next cell, which then fails for a confusing
# secondary reason. Every stage below goes through this instead, so a failure
# stops the notebook where the actual problem is.
import subprocess
import sys


def run(cmd):
    print(f"$ {cmd}\n", flush=True)
    completed = subprocess.run(cmd, shell=True)
    if completed.returncode != 0:
        raise RuntimeError(
            f"\nSTAGE FAILED (exit {completed.returncode}): {cmd}\n"
            f"Read the error above -- later stages depend on this one and will\n"
            f"fail for unrelated-looking reasons if you skip past it."
        )
    print(f"\nOK: {cmd}", flush=True)


print("stage runner ready")

## 3. Repair `data/2024-25/gws/merged_gw.csv`

That file was concatenated without aligning columns by name. FPL added seven `mng_*`
columns in GW22 of 2024-25, so every row from GW22 on carried 49 fields against a
42-field header — and every reader in this project opens it with
`on_bad_lines='skip'`, silently discarding 13,427 rows (49% of the season).

Run with no arguments first to audit every season, then repair.

In [ ]:
run('python scripts/rebuild_merged_gw.py')

In [ ]:
run('python scripts/rebuild_merged_gw.py --season 2024-25 --write')

## 4. Rebuild the dataset

Re-merges every season (final.ipynb cells 3–34) so the recovered rows actually reach
the training data, then carries the FBref defensive columns over from the previous
`all_seasons_data_final.csv` by joining on `(season, element, fixture)`.

The 13,105 recovered rows never had FBref stats merged, so they get zeros and
`has_fbref_defensive=0` — visible rather than silently mixed in.

In [ ]:
run('python scripts/build_dataset.py --write')

## 5. Feature engineering

The RAM-hungry stage: lagged previous-game stats, opponent strength, rolling player
form and context features, ~230 columns out.

If the runtime dies here, restart and re-run from this cell — stage 4's output is
already on disk.

In [ ]:
run('python scripts/build_features.py')

## 6. Train

Runs the notebook's own fixed training cells, so the fixes apply by construction:

- direct models train on the **featured** frame, and `prepare_position_data` raises
  instead of silently dropping features (this is what produced the old single-feature
  models fitted on price alone)
- split is by whole season — train 2016-17…2022-23, validation 2023-24,
  test 2024-25 + 2025-26
- scaler, correlation filter and PCA basis are fitted on the **training fold only**
- hyperparameter search uses `TimeSeriesSplit`, not shuffled `KFold`

Expect the test R² to come in **below** the numbers in the original notebook. The old
figures came from a shuffled split that let the model see neighbouring gameweeks; these
are honest season-holdout numbers.

In [ ]:
run('python scripts/train.py')

## 7. Results

In [ ]:
import json
import os

import pandas as pd

if not os.path.exists('model_metrics.json'):
    raise SystemExit(
        "model_metrics.json is missing, which means section 6 (training) did not\n"
        "finish. Scroll up to section 6 and read the error there -- that is the\n"
        "real problem; this cell is just the first thing to notice it.\n\n"
        "Common causes:\n"
        "  - the runtime ran out of memory during training (check Runtime ->\n"
        "    View resources). Re-run section 6 with: run('python scripts/train.py')\n"
        "  - section 5 never produced all_seasons_data_featured.csv\n"
        f"  - wrong working directory: currently {os.getcwd()}\n"
        "    (after a runtime restart you must re-run section 1)"
    )

metrics = json.load(open('model_metrics.json'))

rows = []
for approach in ('direct', 'pca'):
    for position, res in metrics.get(approach, {}).items():
        for name, m in res['models'].items():
            rows.append({
                'approach': approach,
                'position': position,
                'model': name,
                'train_r2': round(m['train_r2'], 4),
                'val_r2': round(m['val_r2'], 4),
                'test_r2': round(m['test_r2'], 4),
                'test_mae': round(m['test_mae'], 4),
            })

table = pd.DataFrame(rows).sort_values(['approach', 'position', 'test_r2'],
                                       ascending=[True, True, False])
display(table)

print("\nBest per position (by test R2):")
display(table.loc[table.groupby(['approach', 'position']).test_r2.idxmax()])

In [ ]:
# A large train/test gap is the thing to watch: it means the model is fitting
# season-specific noise rather than something that transfers.
table['gap'] = (table.train_r2 - table.test_r2).round(4)
display(table.sort_values('gap', ascending=False).head(10))

## 8. Copy the results back to Drive

`/content/fpl` is wiped when the runtime shuts down, so copy the outputs to Drive
before you close the tab.

Saved: the trained models, scalers, PCA transformers and feature lists under
`saved_models/`, the metrics, and the rebuilt datasets (so you can rerun training
later without redoing stages 3-5).

In [ ]:
import os
import shutil

DEST = '/content/drive/MyDrive/fpl_results'
os.makedirs(DEST, exist_ok=True)

if os.path.exists(f'{DEST}/saved_models'):
    shutil.rmtree(f'{DEST}/saved_models')
shutil.copytree('saved_models', f'{DEST}/saved_models')

for f in ('model_metrics.json', 'pca_model_results.csv',
          'all_seasons_data_final.csv', 'all_seasons_data_featured.csv'):
    if os.path.exists(f):
        shutil.copy2(f, DEST)
        print(f"copied {f} ({os.path.getsize(f) / 1e6:.0f} MB)")

print(f"\neverything is in Drive at {DEST}")
!ls -la $DEST